# 04 · Recycling Instruction Generation (RAG)
**Owner: Eglen**

Part of the EcoSort Waste Management System group project.

## Environment setup (works in **VS Code** and **Google Colab**)
Run this cell first. It detects where you are, gets the code + dataset, and installs deps.
- **VS Code / local:** it uses the repo files already on disk. Put `realwaste.zip` in `data/` (or let it auto-download).
- **Colab:** it clones the repo and downloads the dataset straight to Colab's fast local disk (no Google Drive needed).

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
DATASET_URL = ""  # <-- paste the GitHub *Release* asset URL for realwaste.zip here

if IN_COLAB:
    # 1. get the code
    if not os.path.exists("Ecosort-Waste-Management-System"):
        !git clone https://github.com/REPLACE_ME/Ecosort-Waste-Management-System.git
    %cd Ecosort-Waste-Management-System
    # 2. install deps
    !pip -q install -r requirements.txt
    # 3. get the dataset onto Colab local disk (fast) — not Drive
    if not os.path.exists("data/RealWaste"):
        if DATASET_URL:
            !wget -q -O data/realwaste.zip "$DATASET_URL"
        else:
            print("Set DATASET_URL (GitHub Release asset) or upload data/realwaste.zip")

# make `src` importable from the repo root in both environments
ROOT = os.getcwd()
if os.path.basename(ROOT) == "notebooks":
    ROOT = os.path.dirname(ROOT); os.chdir(ROOT)
sys.path.insert(0, ROOT)

from src import config as C, data_prep
print("Ready. Categories:", C.CATEGORIES)


## Part 4 — Retrieval-Augmented Generation
Embed policy docs + disposal tips, build FAISS index, generate with FLAN-T5.

In [ ]:
from src import rag_system
index,corpus = rag_system.build_index()
print(len(corpus),'retrievable chunks')

In [ ]:
docs = rag_system.retrieve('How do I recycle glass bottles?', k=3, category='Glass')
for d in docs: print('-', d['source'])

In [ ]:
text,docs = rag_system.generate_instructions('Plastic')
print(text)
print('\nSources:',[d['source'] for d in docs])

In [ ]:
# experiment with sampling
for cat in ['Metal','Food Organics','Cardboard']:
    t,_ = rag_system.generate_instructions(cat)
    print(cat,'->',t[:200],'\n')

### TODO (Eglen)
- Experiment with temperature/top_p/max_new_tokens
- Evaluate factual consistency vs the retrieved policy text
- Comment on readability (results are modest by design — grounding is the point)